# Convolution pyramidale HEALPix sur Sentinel-2 réel (niveau 17)

Ce notebook applique `HealPixKernelPyramid`/`HealPixPyramidConv`
(voir `docs/pyramid_convolution.md`) à une vraie acquisition Sentinel-2 L2A,
lue directement depuis le bucket public GRID4EARTH au niveau HEALPix 17
(le niveau publié le plus grossier, donc le moins cher à lire), avec la même
source de données que `dino_tuning_single_date.ipynb`
(`g4e_source.py`, à côté de ce notebook).

**Ce que ce notebook montre** : filtrage multi-échelle NaN-aware (les nuages
et bords de tuile Sentinel-2 sont de vrais trous, pas des trous synthétiques)
sur une donnée réelle, et l'effet du nombre d'étages de la pyramide (`Jmax`)
sur le comblement de ces trous — un noyau `HealPixConv` isolé (bande unique,
sans pyramide) ne peut combler qu'un trou plus petit que son propre gabarit
compact ; les étages plus grossiers de la pyramide voient plus loin.

**Important — non exécuté ici.** Ce notebook a été écrit et relu avec soin,
mais **pas exécuté** : l'environnement cloud utilisé pour développer et
tester `HealPixKernelPyramid`/`HealPixPyramidConv` (voir
`compte_rendu_convolution_pyramidale.md`) n'a pas accès réseau à
`data.grid4earth.eu` (bloqué par la politique réseau de ce bac à sable). À
exécuter dans un environnement qui a accès à ce bucket et aux paquets
`xarray`/`zarr`/`obstore`/`cartopy`/`healpix_plot`/`umap` déjà utilisés par
les notebooks DINO existants (ex. Datarmor, ou tout poste où
`dino_tuning_single_date.ipynb` fonctionne déjà). Si une cellule échoue, ce
n'est donc pas une régression connue et corrigée — c'est la première vraie
exécution.


## 1. Paramètres

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # pour importer g4e_source (à côté de ce notebook)
sys.path.insert(0, str(Path.cwd().parent))   # pour importer healpix_analyse en dev

from g4e_source import G4E_L2A, G4E_PRODUCTS, RGB, ProductSeries

from healpix_analyse.decomp import HealPixDecomp
from healpix_analyse.kernel_pyramid import HealPixKernelPyramid, kernel_gaussian
from healpix_analyse.pyramid_conv import HealPixPyramidConv
from healpix_analyse.dino import nested_to_tiles

# --------------------------------------------------------------------------
# À RÉGLER
# --------------------------------------------------------------------------
PRODUCTS    = list(G4E_PRODUCTS)     # mêmes produits que dino_tuning_single_date.ipynb
TIME_INDEX  = 0                      # quel produit de PRODUCTS lire
DATA_LEVEL  = 17                     # niveau HEALPix publié le plus grossier (17, 19 ou 20)
SCALING     = "reflectance"          # "reflectance" (/10000), "percentile", "none"
CACHE       = os.path.expanduser("~/s2_cache")   # None pour ne rien mettre en cache

JMAX               = 4    # nombre d'étages Down de la pyramide (voir §7 pour l'effet de ce choix)
COMPACT_KERNEL_SZ  = 5    # taille (impaire) du noyau HealPixConv par bande
SIGMA_PIX          = 1.2  # largeur du noyau gaussien, en pixels de CHAQUE bande (docs/pyramid_convolution.md §A.2)
GAUGE_TYPE         = "phi"  # jauge simple ; une scène Sentinel-2 est loin des pôles géographiques

DTYPE  = torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", DEVICE)


## 2. Lecture d'une acquisition réelle (GRID4EARTH, niveau 17)

Identique au §2 de `dino_tuning_single_date.ipynb` : chaque produit est un
store zarr séparé, le niveau HEALPix y est un **groupe**
(`measurements/reflectance/<niveau>`), et la première lecture télécharge le
produit (les suivantes relisent le cache disque si `CACHE` est renseigné).


In [ ]:
src = ProductSeries(PRODUCTS, level=DATA_LEVEL, base=G4E_L2A, bands=RGB, scaling=SCALING)
level, cell_id, dates = src.level, src.cell_id, src.dates

print(f"niveau HEALPix {level}, {cell_id.size} cellules, produit {dates[TIME_INDEX]}")
print("ellipsoïde déclaré par le store :", src.ellipsoid)

t0 = time.time()
rgb = src.rgb(TIME_INDEX, cache=CACHE)     # [N, 3] réflectances dans [0, 1] ; NaN = absent/masqué
print(f"lu en {time.time() - t0:.0f}s ; NaN : {100 * np.isnan(rgb).any(1).mean():.1f}% des cellules")


## 3. Pyramide et noyau pyramidal, alignés sur la géométrie réelle du store

Point important (voir `docs/pyramid_convolution.md`, §A.5) : ce module
utilise `ellipsoid="sphere"` par défaut, mais les produits EOPF déclarent en
général `"wgs84"`. On passe explicitement `ellipsoid=src.ellipsoid` à
`HealPixDecomp` **et** à `HealPixKernelPyramid.from_kernel`, pour que les
deux utilisent la même géométrie que les cellules réelles du store — sans
quoi les distances seraient silencieusement mal étiquetées.


In [ ]:
decomp = HealPixDecomp(
    level=level, cell_ids=cell_id, Jmax=JMAX,
    ellipsoid=src.ellipsoid, dtype=DTYPE, device=DEVICE,
)
print(decomp)
print("tailles des bandes (fine -> grossière) :", decomp.sizes)

t0 = time.time()
kernel_pyramid = HealPixKernelPyramid.from_kernel(
    decomp, kernel_gaussian(sigma_pix=SIGMA_PIX),
    compact_kernel_sz=COMPACT_KERNEL_SZ, gauge_type=GAUGE_TYPE,
    ellipsoid=src.ellipsoid, dtype=DTYPE,
)
print(f"noyau pyramidal construit en {time.time() - t0:.0f}s ({decomp.n_bands} bandes)")

pconv = HealPixPyramidConv(decomp, kernel_pyramid, mode="normalized")


## 4. Convolution pyramidale masquée sur les 3 bandes

In [ ]:
# [N, 3] -> [3, N] : chaque bande de couleur est un échantillon de "batch"
# passé au même noyau à 1 canal (voir docs/pyramid_convolution.md §C).
x = torch.as_tensor(rgb.T, dtype=DTYPE, device=DEVICE)

t0 = time.time()
with torch.no_grad():
    y, support = pconv(x, return_support=True)
print(f"convolution pyramidale (3 bandes) en {time.time() - t0:.0f}s")

y_np = y.detach().cpu().numpy() if torch.is_tensor(y) else np.asarray(y)
support_np = support.detach().cpu().numpy() if torch.is_tensor(support) else np.asarray(support)
rgb_filtered = y_np.T                 # [N, 3]
support_map = support_np[0]           # [N] -- identique pour les 3 bandes (même masque de départ)

nan_before = 100 * np.isnan(rgb).any(1).mean()
nan_after = 100 * np.isnan(rgb_filtered).any(1).mean()
print(f"NaN avant filtrage                    : {nan_before:.1f}% des cellules")
print(f"NaN après (pyramide, Jmax={JMAX})        : {nan_after:.1f}% des cellules")
print(f"support synthétisé : min {np.nanmin(support_map):.3g}, "
      f"max {np.nanmax(support_map):.3g}, "
      f"médiane (où fini) {np.nanmedian(support_map):.3g}")


## 5. Comparaison visuelle : tuiles avant / après

Même repliage en tuiles que les notebooks DINO (`nested_to_tiles`), pour
regarder de près ce que le filtrage a réellement fait sur des vraies données
(pas seulement des statistiques globales).


In [ ]:
def to_rgb(tile, p_low=2, p_high=98):
    """[3, S, S] -> image affichable, étirée sur des percentiles (NaN en noir)."""
    img = np.transpose(tile, (1, 2, 0))
    finite = img[np.isfinite(img)]
    if finite.size == 0:
        return np.zeros_like(img)
    lo, hi = np.nanpercentile(finite, [p_low, p_high])
    return np.clip((np.nan_to_num(img, nan=lo) - lo) / max(hi - lo, 1e-6), 0, 1)


TILE_LEVELS = 8   # tuiles de 2**TILE_LEVELS px ; augmenter si les vignettes ci-dessous sont trop petites
parent_level = level - TILE_LEVELS
if parent_level < 0:
    raise ValueError(f"TILE_LEVELS={TILE_LEVELS} > level={level} ; réduire TILE_LEVELS")

tiles_before, parent_ids, coverage, _ = nested_to_tiles(rgb, cell_id, level, parent_level, fill="nan")
tiles_after, parent_ids2, coverage2, _ = nested_to_tiles(rgb_filtered, cell_id, level, parent_level, fill="nan")
assert np.array_equal(parent_ids, parent_ids2), "les deux repliages doivent couvrir les mêmes tuiles"

order = np.argsort(-coverage)[:6]   # les tuiles les mieux couvertes d'abord (les plus lisibles)
fig, axes = plt.subplots(2, len(order), figsize=(2.4 * len(order), 5.2), squeeze=False)
for j, i in enumerate(order):
    axes[0, j].imshow(to_rgb(tiles_before[i]))
    axes[0, j].set_title(f"avant  cov={coverage[i]:.2f}", fontsize=8)
    axes[1, j].imshow(to_rgb(tiles_after[i]))
    axes[1, j].set_title("après (pyramide)", fontsize=8)
    for a in axes[:, j]:
        a.set_axis_off()
fig.tight_layout()
plt.show()


## 6. Carte HEALPix complète

In [ ]:
import cartopy.crs as ccrs
import healpix_plot

grid = healpix_plot.HealpixGrid(level=level, indexing_scheme="nested", ellipsoid=src.ellipsoid)

fig, axes = plt.subplots(1, 3, figsize=(20, 6),
                          subplot_kw={"projection": ccrs.PlateCarree()}, layout="constrained")
hi = float(np.nanpercentile(rgb, 98))
healpix_plot.plot(cell_id, rgb, healpix_grid=grid, sampling_grid={"shape": 900},
                   ax=axes[0], rgb_clip=(0.0, hi), axis_labels="none",
                   title=f"{dates[TIME_INDEX]}  RGB brut (niveau {level})")
healpix_plot.plot(cell_id, rgb_filtered, healpix_grid=grid, sampling_grid={"shape": 900},
                   ax=axes[1], rgb_clip=(0.0, hi), axis_labels="none",
                   title=f"après convolution pyramidale (Jmax={JMAX}, {COMPACT_KERNEL_SZ}x{COMPACT_KERNEL_SZ})")
mp = healpix_plot.plot(cell_id, support_map, healpix_grid=grid, sampling_grid={"shape": 900},
                        ax=axes[2], axis_labels="none",
                        title="support synthétisé (confiance après filtrage)")
fig.colorbar(mp, ax=axes[2], shrink=0.7)
plt.show()


## 7. Effet du nombre d'étages (`Jmax`) sur le comblement des trous

Un noyau `HealPixConv` isolé (une seule bande, `Jmax=0` revient à ça) ne peut
combler qu'un trou plus petit que son propre gabarit compact
(`COMPACT_KERNEL_SZ`). Les étages plus grossiers de la pyramide (`Jmax` plus
grand) voient un contexte spatial plus large à chaque étage supplémentaire, au
prix d'un noyau supplémentaire à construire. On mesure ici, sur les vrais
trous de cette acquisition (nuages, bord de tuile), la fraction de cellules
qui obtiennent un support non nul après synthèse — sans regarder les valeurs,
juste "est-ce qu'il y a une réponse du tout".


In [ ]:
JMAX_VALUES = sorted(set([0, 1, 2, JMAX]))
x1 = x[:1]   # une seule bande suffit : les 3 bandes partagent le même masque

coverage_by_jmax = {}
for jm in JMAX_VALUES:
    d = HealPixDecomp(level=level, cell_ids=cell_id, Jmax=jm,
                       ellipsoid=src.ellipsoid, dtype=DTYPE, device=DEVICE)
    kp = HealPixKernelPyramid.from_kernel(
        d, kernel_gaussian(SIGMA_PIX), compact_kernel_sz=COMPACT_KERNEL_SZ,
        gauge_type=GAUGE_TYPE, ellipsoid=src.ellipsoid, dtype=DTYPE,
    )
    pc = HealPixPyramidConv(d, kp, mode="normalized")
    with torch.no_grad():
        _, supp = pc(x1, return_support=True)
    supp_np = supp.detach().cpu().numpy() if torch.is_tensor(supp) else np.asarray(supp)
    frac_supported = float((supp_np[0] > 1e-6).mean())
    coverage_by_jmax[jm] = frac_supported
    print(f"Jmax={jm:>2d} ({d.n_bands} bande(s)) : {100 * frac_supported:.1f}% des cellules ont un support > 0")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(list(coverage_by_jmax.keys()), [100 * v for v in coverage_by_jmax.values()], "o-")
ax.set_xlabel("Jmax (étages de la pyramide)")
ax.set_ylabel("% cellules avec support > 0")
ax.set_title("Comblement des trous réels vs profondeur de la pyramide")
fig.tight_layout()
plt.show()


## Notes finales

- Ce notebook réutilise volontairement `g4e_source.py` tel quel (même
  lecture, même mise en cache, même normalisation) pour rester cohérent avec
  les notebooks DINO existants — aucune donnée n'est reprojetée ou
  rééchantillonnée en dehors de ce que `HealPixDecomp`/`HealPixConv` font
  déjà.
- Le choix `ellipsoid=src.ellipsoid` (§3) est important et documenté comme
  limite ouverte dans `docs/pyramid_convolution.md` (§A.5, §E point 5) : rien
  ne vérifie automatiquement qu'un `HealPixDecomp`/`HealPixKernelPyramid`
  construits avec des ellipsoïdes différents sont utilisés ensemble par
  erreur.
- `JMAX`, `COMPACT_KERNEL_SZ`, `SIGMA_PIX` et `GAUGE_TYPE` sont volontairement
  regroupés en tête de notebook (§1) pour être rejoués sans tout relire ; la
  lecture réseau (§2) est ce qui coûte le plus cher, d'où le cache disque.
- Pas de comparaison ici contre l'oracle indépendant de
  `healpix_analyse.validation` (déjà fait sur données synthétiques dans
  `tests/test_kernel_pyramid.py` et documenté dans
  `docs/pyramid_convolution.md` §E) : ce notebook est une démonstration sur
  données réelles, pas une nouvelle campagne de validation numérique.
